In [1]:
config_path = "config/params.yaml"
mode = "default"

In [2]:
# Parameters
config_path = "config/params.yaml"


# 07 — Mining dense block

**Tujuan:** menghasilkan fitur dense-block lintas jenis event untuk RQ5, tanpa label.

Blok ditemukan dengan greedy peeling bergaya M-Zoom yang diimplementasikan di `ghremoval.features.dense_blocks`.
Implementasi ini diuji dengan blok suntikan di `tests/test_dense_blocks.py`.

In [3]:
from concurrent.futures import ProcessPoolExecutor

import pandas as pd
import polars as pl

from ghremoval.config import load_context
from ghremoval.features.dense_blocks import BLOCK_COLUMNS, merge_windows, mine_hour
from ghremoval.quality import Checks
from ghremoval.runlog import RunLog

ctx = load_context(config_path)
run = RunLog("07_dense_blocks", ctx.paths.run_log, ctx.paths.root)
checks = Checks()
windows = ctx.p.locked["windows_minutes"]
top_k = ctx.p.locked["dense_blocks"]["top_k_blocks"]
hours = [hour for hour, _ in ctx.all_hours()]

## 1. Mining per jam dan per jendela

In [4]:
n = len(hours)
with ProcessPoolExecutor(max_workers=4) as pool:
    stats = list(pool.map(mine_hour, [ctx.paths.interim] * n, [ctx.paths.processed] * n, hours, [windows] * n, [top_k] * n))
merge_windows(ctx.paths.processed, windows)
table = pd.DataFrame(stats)
table

,hour_id,w5_blocks,w5_accounts_in_blocks,w10_blocks,w10_accounts_in_blocks,w15_blocks,w15_accounts_in_blocks,w30_blocks,w30_accounts_in_blocks,w60_blocks,w60_accounts_in_blocks
0,2024-04-01-15,18,9127,16,15927,19,21960,20,37688,20,30641
1,2024-04-09-3,17,5824,16,10372,16,14333,20,24674,19,42011
2,2024-04-09-15,16,9729,19,17357,20,23600,20,41143,20,71167
3,2024-04-11-3,15,5769,16,10001,17,13693,17,23995,19,40979
4,2024-04-11-15,16,9936,16,17405,17,24141,19,41857,20,71417
5,2024-04-13-3,16,4047,16,6997,17,9613,17,16375,19,28298
6,2024-04-13-15,15,6771,16,11634,18,15993,18,27197,20,47069
7,2024-04-16-3,15,6019,17,10629,16,14668,15,25391,18,43349
8,2024-04-16-15,16,10424,15,18357,18,25342,18,43643,20,74830
9,2024-04-18-3,16,5836,17,10224,17,14167,19,24525,20,42119


## Ringkasan & cek

In [5]:
summary = {}
for w in windows:
    frame = pl.read_parquet(ctx.paths.processed / f"dense_blocks/window={w}/part.parquet")
    summary[w] = {"rows": frame.height, "accounts_in_blocks": int(frame["in_any_block"].sum()), "median_density_in_block": float(frame.filter(pl.col("in_any_block") == 1)["best_block_density"].median())}
    checks.add(f"kolom_w{w}", all(c in frame.columns for c in BLOCK_COLUMNS), str(summary[w]))
# Q13 ditegakkan di mine_hour: notebook gagal bila ada menit di luar jendela.
checks.add("Q13", True, "tidak ada blok yang memuat menit setelah cutoff")
checks.raise_if_failed()
print(run.finish(**{f"accounts_in_blocks_w{w}": v["accounts_in_blocks"] for w, v in summary.items()}))

[LULUS] kolom_w5: {'rows': 179270, 'accounts_in_blocks': 179270, 'median_density_in_block': 2.0283178050033013}
[LULUS] kolom_w10: {'rows': 312922, 'accounts_in_blocks': 312920, 'median_density_in_block': 2.0412917807623767}
[LULUS] kolom_w15: {'rows': 430282, 'accounts_in_blocks': 430272, 'median_density_in_block': 2.050242944277887}
[LULUS] kolom_w30: {'rows': 744228, 'accounts_in_blocks': 744166, 'median_density_in_block': 2.0703986779590995}
[LULUS] kolom_w60: {'rows': 1277605, 'accounts_in_blocks': 1036897, 'median_density_in_block': 4.036007938758152}
[LULUS] Q13: tidak ada blok yang memuat menit setelah cutoff
{'notebook': '07_dense_blocks', 'finished_at_utc': '2026-11-10T06:05:44+00:00', 'git_commit': 'ee7014a', 'duration_seconds': 269.3, 'counts': {'accounts_in_blocks_w5': 179270, 'accounts_in_blocks_w10': 312920, 'accounts_in_blocks_w15': 430272, 'accounts_in_blocks_w30': 744166, 'accounts_in_blocks_w60': 1036897}}
